<h2><b>Composed Image Retrieval for Fashion</b></h2>
<h3>Computer Vision Spring 2026 Final Project</h3>
<h6>20235227 Bereket Yimolal Tiruneh </br>20235218 Abishev Temirlan </br>20261084 박상원
</h6>

In [1]:
!pip install -q transformers==4.41.0 peft==0.11.1 faiss-cpu gradio ftfy pillow tqdm huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 30.7 MB/s eta 0:00:00


In [2]:
# 2. Core Imports & Device Setup
import os
import zipfile
import shutil
import torch
import json
import numpy as np
import faiss
import gradio as gr
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model
from google.colab import drive
from huggingface_hub import hf_hub_download

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [3]:
# 3. Path Configuration, Mount Google Drive & Extract Dataset
# Centralized Paths (Drive is mounted for FAISS index and Embedding JSON storage)
DRIVE_BASE = '/content/drive/MyDrive/CV_final' #@param{type:\"string\"}
DRIVE_CKPT = f'{DRIVE_BASE}/checkpoints'
LOCAL_EXTRACT_DIR = '/content/fashion_iq_dataset'

# ==========================================
# ⚙️ HUGGING FACE REPOSITORY CONFIGURATION
# ==========================================
HF_MODEL_REPO = "berekety/fashion-iq-lora-clip"
HF_DATASET_REPO = "berekety/fashion-iq-data"
FORCE_RETRAIN = False
# ==========================================

try:
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    print("Drive already mounted or mountpoint contains files. Proceeding...")

# Pull dataset zip from Hugging Face and extract locally so Gradio can read physical files
if not os.path.exists(LOCAL_EXTRACT_DIR):
    print("📥 Pulling fashion_iq_dataset-1.zip from Hugging Face Hub...")
    try:
        hf_zip_target = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename="fashion_iq_dataset-1.zip",
            repo_type="dataset"
        )
        print("🔓 Unzipping dataset locally to runtime storage...")
        with zipfile.ZipFile(hf_zip_target, 'r') as zip_ref:
            zip_ref.extractall("/content/")
        print("Fresh extraction complete!")
    except Exception as e:
        print(f"Failed to fetch dataset from Hugging Face: {e}")
else:
    print("Dataset cache folder detected locally. Skipping extraction.")

Mounted at /content/drive
📥 Pulling fashion_iq_dataset-1.zip from Hugging Face Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


fashion_iq_dataset-1.zip:   0%|          | 0.00/838M [00:00<?, ?B/s]

🔓 Unzipping dataset locally to runtime storage...
Fresh extraction complete!


In [4]:
# 4. Transforms & Dataset Definition
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

class LiveCIRDataset(Dataset):
    def __init__(self, caption_json_path, img_dir, transform=None):
        with open(caption_json_path, 'r') as f:
            self.triples = json.load(f)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        item = self.triples[idx]
        ref_asin = item['candidate']
        tgt_asin = item['target']
        caption = " and ".join(item['captions']) if isinstance(item['captions'], list) else item['captions']

        ref_path = os.path.join(self.img_dir, f"{ref_asin}.jpg")
        tgt_path = os.path.join(self.img_dir, f"{tgt_asin}.jpg")

        try:
            ref_img = Image.open(ref_path).convert('RGB')
            tgt_img = Image.open(tgt_path).convert('RGB')
        except Exception:
            ref_img = Image.new('RGB', (224, 224), color=0)
            tgt_img = Image.new('RGB', (224, 224), color=0)

        if self.transform:
            ref_img = self.transform(ref_img)
            tgt_img = self.transform(tgt_img)

        return {'ref_img': ref_img, 'tgt_img': tgt_img, 'caption': caption}

In [5]:
# 5. Model Architecture
def get_lora_cir_backbone(model_name="openai/clip-vit-base-patch32"):
    base_model = CLIPModel.from_pretrained(model_name).to(device)
    processor = CLIPProcessor.from_pretrained(model_name)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
        lora_dropout=0.1,
        bias="none"
    )

    lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters()
    return lora_model, processor

class DeepCrossAttentionCombiner(nn.Module):
    def __init__(self, dim=512, num_heads=8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.norm_txt = nn.LayerNorm(dim)
        self.norm_img = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, dim * 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(dim * 2, dim)
        )
        self.gate = nn.Sequential(nn.Linear(dim * 2, 1), nn.Sigmoid())

    def forward(self, ref_patches, txt_tokens, ref_cls, txt_cls):
        attended_features, _ = self.cross_attn(txt_tokens, ref_patches, ref_patches)
        txt_enriched = self.norm_txt(txt_tokens + attended_features)
        multimodal_seq_summary = txt_enriched.mean(dim=1)

        global_concat = torch.cat([ref_cls, txt_cls], dim=-1)
        learned_combination = self.mlp(global_concat)

        alpha = self.gate(global_concat)
        composed_query = alpha * learned_combination + (1 - alpha) * multimodal_seq_summary

        return F.normalize(composed_query, dim=-1)

class GlobalInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, composed_queries, target_images):
        queries = F.normalize(composed_queries, dim=-1)
        targets = F.normalize(target_images, dim=-1)
        logits = torch.matmul(queries, targets.T) / self.temperature
        labels = torch.arange(logits.shape[0], dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [6]:
# 6. Training Pipeline (Hugging Face Cache-First with Local Drive Fallback)
def train_cir_system(categories=['dress', 'shirt', 'toptee'], epochs=10, batch_size=32):
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    CLIP_CKPT = f'{DRIVE_CKPT}/lora_clip_final.pt'
    COMBINER_CKPT = f'{DRIVE_CKPT}/combiner_head_final.pt'

    lora_clip, processor = get_lora_cir_backbone()
    combiner = DeepCrossAttentionCombiner(dim=512).to(device)

    # Check Hugging Face Hub first for fine-tuned weights
    if not FORCE_RETRAIN:
        print(f"🔎 Checking Hugging Face ({HF_MODEL_REPO}) for pre-trained weights...")
        try:
            dl_clip = hf_hub_download(repo_id=HF_MODEL_REPO, filename="lora_clip_final.pt")
            dl_combiner = hf_hub_download(repo_id=HF_MODEL_REPO, filename="combiner_head_final.pt")

            lora_clip.load_state_dict(torch.load(dl_clip, map_location=device))
            combiner.load_state_dict(torch.load(dl_combiner, map_location=device))
            lora_clip.eval()
            combiner.eval()
            print("Pre-trained weights loaded successfully from Hugging Face! Skipping training loop.")
            return lora_clip, combiner, processor
        except Exception as e:
            print(f"Could not pull weights from Hugging Face: {e}")
            print("Checking Google Drive local path for checkpoints...")

            if os.path.exists(CLIP_CKPT) and os.path.exists(COMBINER_CKPT):
                print("⚡ Found saved checkpoints on local Drive. Loading — skipping training...")
                lora_clip.load_state_dict(torch.load(CLIP_CKPT, map_location=device))
                combiner.load_state_dict(torch.load(COMBINER_CKPT, map_location=device))
                lora_clip.eval()
                combiner.eval()
                return lora_clip, combiner, processor

    print("🚀 No checkpoints found on HF or Drive. Starting active training loop...")
    criterion = GlobalInfoNCELoss(temperature=0.07)
    optimizer = torch.optim.AdamW(list(lora_clip.parameters()) + list(combiner.parameters()), lr=5e-5, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    loaders = []
    for cat in categories:
        json_p = f"{LOCAL_EXTRACT_DIR}/captions/caption/cap.{cat}.train.json"
        img_p  = f"{LOCAL_EXTRACT_DIR}/images/{cat}"
        if os.path.exists(json_p):
            ds = LiveCIRDataset(json_p, img_p, transform=train_transform)
            loaders.append(DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2))

    if not loaders:
        raise ValueError("❌ No data loaders could be created. Check dataset extraction.")

    print(f"🚀 Live training pipeline active...")
    for epoch in range(epochs):
        lora_clip.train()
        combiner.train()
        total_loss, total_batches = 0, 0

        for loader in loaders:
            for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
                text_inputs = processor(text=batch['caption'], return_tensors="pt", padding=True, truncation=True).to(device)
                ref_pixels = batch['ref_img'].to(device)
                tgt_pixels = batch['tgt_img'].to(device)

                txt_outputs = lora_clip.text_model(**text_inputs)
                txt_tokens  = lora_clip.text_projection(txt_outputs.last_hidden_state)
                txt_cls     = F.normalize(txt_tokens[:, 0, :], dim=-1)

                ref_outputs = lora_clip.vision_model(pixel_values=ref_pixels)
                ref_patches = lora_clip.visual_projection(ref_outputs.last_hidden_state)
                ref_cls     = F.normalize(ref_patches[:, 0, :], dim=-1)

                tgt_outputs = lora_clip.vision_model(pixel_values=tgt_pixels)
                tgt_features = lora_clip.visual_projection(tgt_outputs.pooler_output)
                tgt_features = F.normalize(tgt_features, dim=-1)

                composed_queries = combiner(ref_patches, txt_tokens, ref_cls, txt_cls)
                loss = criterion(composed_queries, tgt_features)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(list(lora_clip.parameters()) + list(combiner.parameters()), max_norm=1.0)
                optimizer.step()

                total_loss += loss.item()
                total_batches += 1

        scheduler.step()
        avg_loss = total_loss / total_batches if total_batches > 0 else 0
        print(f"📉 Epoch {epoch+1} Completed — Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    torch.save(lora_clip.state_dict(), CLIP_CKPT)
    torch.save(combiner.state_dict(), COMBINER_CKPT)
    print("✅ Model trained successfully and weights saved to Drive.")
    return lora_clip, combiner, processor

trained_clip, trained_combiner, active_processor = train_cir_system()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

trainable params: 983,040 || all params: 152,260,353 || trainable%: 0.6456
🔎 Checking Hugging Face (berekety/fashion-iq-lora-clip) for pre-trained weights...


lora_clip_final.pt:   0%|          | 0.00/609M [00:00<?, ?B/s]

combiner_head_final.pt:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

Pre-trained weights loaded successfully from Hugging Face! Skipping training loop.


In [ ]:
# 7. Validation Evaluation Setup (Hugging Face Native with Category Breakdown)


@torch.no_grad()
def evaluate_recall(lora_clip, combiner, processor, categories=['dress', 'shirt', 'toptee']):
    global eval_index, eval_gallery_paths
    lora_clip.eval()
    combiner.eval()

    # 1. Smart Fetch from Hugging Face
    if not FORCE_RETRAIN:
        print(f"🔎 Checking Hugging Face ({HF_MODEL_REPO}) for Eval Index...")
        try:
            dl_faiss = hf_hub_download(repo_id=HF_MODEL_REPO, filename="eval_index.faiss")
            dl_paths = hf_hub_download(repo_id=HF_MODEL_REPO, filename="eval_gallery_paths.json")

            eval_index = faiss.read_index(dl_faiss)
            with open(dl_paths, 'r') as f:
                eval_gallery_paths = json.load(f)

            print(f"📊 Eval gallery loaded successfully from HF! Size: {eval_index.ntotal}")
            index_ready = True
        except Exception as e:
            print(f"⚠️ Could not fetch Eval Index from HF: {e}")
            index_ready = False
    else:
        index_ready = False

    # 2. Fallback: Build locally
    if not index_ready:
        print("🔄 Building val-only gallery maps locally...")
        gallery_feats, eval_gallery_paths = [], []

        for cat in categories:
            split_p = f"{LOCAL_EXTRACT_DIR}/image_splits/split.{cat}.val.json"
            img_dir = f"{LOCAL_EXTRACT_DIR}/images/{cat}"
            if not os.path.exists(split_p): continue
            with open(split_p, 'r') as f:
                asins = json.load(f)

            for asin in tqdm(asins, desc=f"Indexing val/{cat}"):
                img_path = os.path.join(img_dir, f"{asin}.jpg")
                if not os.path.exists(img_path): continue
                img = Image.open(img_path).convert('RGB')
                tensor_img = val_transform(img).unsqueeze(0).to(device)

                v_out = lora_clip.vision_model(pixel_values=tensor_img)
                feat = lora_clip.visual_projection(v_out.pooler_output)
                feat = F.normalize(feat, dim=-1).cpu().numpy().flatten()
                gallery_feats.append(feat)
                eval_gallery_paths.append(img_path)

        gallery_matrix = np.vstack(gallery_feats).astype('float32')
        eval_index = faiss.IndexFlatIP(512)
        eval_index.add(gallery_matrix)
        print(f"💾 New eval index compiled locally. Size: {eval_index.ntotal}")

    # 3. Execute Recall Evaluation
    asin_to_idx = {Path(p).stem: idx for idx, p in enumerate(eval_gallery_paths)}
    k_values = [1, 10, 20, 50, 100]

    # Nested dictionary to isolate hits and queries for each garment type
    category_metrics = {
        cat: {"hits_at_k": {k: 0 for k in k_values}, "total_queries": 0}
        for cat in categories
    }

    for cat in categories:
        json_p = f"{LOCAL_EXTRACT_DIR}/captions/caption/cap.{cat}.val.json"
        if not os.path.exists(json_p): continue
        with open(json_p, 'r') as f:
            val_triples = json.load(f)

        for item in val_triples:
            ref_asin, tgt_asin = item['candidate'], item['target']
            if ref_asin not in asin_to_idx or tgt_asin not in asin_to_idx: continue
            caption = " and ".join(item['captions']) if isinstance(item['captions'], list) else item['captions']

            ref_img_path = eval_gallery_paths[asin_to_idx[ref_asin]]
            try:
                ref_img = val_transform(Image.open(ref_img_path).convert('RGB')).unsqueeze(0).to(device)
            except Exception:
                continue

            text_in = processor(text=[caption], return_tensors="pt", padding=True, truncation=True).to(device)

            txt_outputs = lora_clip.text_model(**text_in)
            txt_tokens  = lora_clip.text_projection(txt_outputs.last_hidden_state)
            txt_cls     = F.normalize(txt_tokens[:, 0, :], dim=-1)

            ref_outputs = lora_clip.vision_model(pixel_values=ref_img)
            ref_patches = lora_clip.visual_projection(ref_outputs.last_hidden_state)
            ref_cls     = F.normalize(ref_patches[:, 0, :], dim=-1)

            query = combiner(ref_patches, txt_tokens, ref_cls, txt_cls).cpu().numpy().astype('float32')

            _, I = eval_index.search(query, k=105)
            retrieved_indices = [int(idx) for idx in I[0] if eval_gallery_paths[idx] != ref_img_path]

            target_global_idx = asin_to_idx[tgt_asin]

            # Route scores to the specific active category metrics profile
            for k in k_values:
                if target_global_idx in retrieved_indices[:k]:
                    category_metrics[cat]["hits_at_k"][k] += 1

            category_metrics[cat]["total_queries"] += 1

    # 4. Print Detailed Metrics & Aggregate for Global Stats
    global_hits_at_k = {k: 0 for k in k_values}
    global_total_queries = 0

    print("\n============================================================")
    print("📊 CATEGORY-SPECIFIC PERFORMANCE BREAKDOWN:")
    print("============================================================")

    for cat in categories:
        cat_total = category_metrics[cat]["total_queries"]
        print(f"\n🛍️ Category: {cat.upper()} ({cat_total} queries processed)")

        if cat_total > 0:
            for k in k_values:
                cat_hits = category_metrics[cat]["hits_at_k"][k]
                print(f"   🎯 Recall@{k:<3} : {(cat_hits / cat_total) * 100:.2f}%")
                # Cumulate global values from underlying category structures
                global_hits_at_k[k] += cat_hits
        else:
            print("   ⚠️ No validation files processed for this target.")

        global_total_queries += cat_total

    print("\n============================================================")
    print(f"🌍 GLOBAL PERFORMANCE SUMMARY ({global_total_queries} total queries):")
    print("============================================================")
    if global_total_queries > 0:
        for k in k_values:
            print(f"   🎯 Recall@{k:<3} : {(global_hits_at_k[k] / global_total_queries) * 100:.2f}%")
    print("============================================================")

evaluate_recall(trained_clip, trained_combiner, active_processor)

🔎 Checking Hugging Face (berekety/fashion-iq-lora-clip) for Eval Index...
📊 Eval gallery loaded successfully from HF! Size: 13138

📊 CATEGORY-SPECIFIC PERFORMANCE BREAKDOWN:

🛍️ Category: DRESS (1845 queries processed)
   🎯 Recall@1   : 5.96%
   🎯 Recall@10  : 24.50%
   🎯 Recall@20  : 33.12%
   🎯 Recall@50  : 45.91%
   🎯 Recall@100 : 57.07%

🛍️ Category: SHIRT (1172 queries processed)
   🎯 Recall@1   : 7.68%
   🎯 Recall@10  : 24.32%
   🎯 Recall@20  : 32.94%
   🎯 Recall@50  : 48.29%
   🎯 Recall@100 : 59.81%

🛍️ Category: TOPTEE (1557 queries processed)
   🎯 Recall@1   : 9.06%
   🎯 Recall@10  : 28.52%
   🎯 Recall@20  : 37.57%
   🎯 Recall@50  : 52.34%
   🎯 Recall@100 : 63.33%

🌍 GLOBAL PERFORMANCE SUMMARY (4574 total queries):
   🎯 Recall@1   : 7.46%
   🎯 Recall@10  : 25.82%
   🎯 Recall@20  : 34.59%
   🎯 Recall@50  : 48.71%
   🎯 Recall@100 : 59.90%


In [7]:
# 8. Smart Demo Index Setup (Hugging Face Native)

@torch.no_grad()
def build_demo_gallery(lora_clip, processor, categories=['dress', 'shirt', 'toptee']):
    global index, gallery_paths
    lora_clip.eval()

    # 1. Try fetching from Hugging Face Hub first
    if not FORCE_RETRAIN:
        print(f"🔎 Checking Hugging Face ({HF_MODEL_REPO}) for Demo Index & Paths...")
        try:
            dl_faiss = hf_hub_download(repo_id=HF_MODEL_REPO, filename="demo_index_final.faiss")
            dl_paths = hf_hub_download(repo_id=HF_MODEL_REPO, filename="demo_gallery_paths_final.json")

            index = faiss.read_index(dl_faiss)
            with open(dl_paths, 'r') as f:
                gallery_paths = json.load(f)

            print(f"📊 Demo index loaded successfully from HF! Gallery size: {index.ntotal}")
            return
        except Exception as e:
            print(f"⚠️ Could not fetch Index from HF: {e}")
            print("🔄 Defaulting to local build fallback...")

    # 2. Fallback: Build from scratch if HF fails or FORCE_RETRAIN is True
    print("🔄 Compiling new full gallery (train + val)...")
    gallery_feats, gallery_paths = [], []

    for split in ['val', 'train']:
        for cat in categories:
            split_p = f"{LOCAL_EXTRACT_DIR}/image_splits/split.{cat}.{split}.json"
            img_dir = f"{LOCAL_EXTRACT_DIR}/images/{cat}"
            if not os.path.exists(split_p): continue
            with open(split_p, 'r') as f:
                asins = json.load(f)

            for asin in tqdm(asins, desc=f"Indexing {split}/{cat}"):
                img_path = os.path.join(img_dir, f"{asin}.jpg")
                if not os.path.exists(img_path): continue
                img = Image.open(img_path).convert('RGB')
                tensor_img = val_transform(img).unsqueeze(0).to(device)

                v_out = lora_clip.vision_model(pixel_values=tensor_img)
                feat = lora_clip.visual_projection(v_out.pooler_output)
                feat = F.normalize(feat, dim=-1).cpu().numpy().flatten()
                gallery_feats.append(feat)
                gallery_paths.append(img_path)

    gallery_matrix = np.vstack(gallery_feats).astype('float32')
    index = faiss.IndexFlatIP(512)
    index.add(gallery_matrix)

    print(f"💾 Demo index successfully compiled locally. Gallery size: {index.ntotal}")

build_demo_gallery(trained_clip, active_processor)

🔎 Checking Hugging Face (berekety/fashion-iq-lora-clip) for Demo Index & Paths...


demo_index_final.faiss:   0%|          | 0.00/107M [00:00<?, ?B/s]

demo_gallery_paths_final.json: 0.00B [00:00, ?B/s]

📊 Demo index loaded successfully from HF! Gallery size: 52464


In [ ]:
 # 9. Gradio Interface Runtime Loop
@torch.no_grad()
def process_visual_query(reference_image, modification_text, top_k):
    if reference_image is None or not modification_text:
        return "⚠️ Please upload a reference image and type a modification query.", []

    try:
        ref_tensor = val_transform(reference_image.convert('RGB')).unsqueeze(0).to(device)
        text_tokenized = active_processor(text=[modification_text], return_tensors="pt", padding=True, truncation=True).to(device)

        trained_clip.eval()
        trained_combiner.eval()

        txt_out = trained_clip.text_model(**text_tokenized)
        txt_tokens = trained_clip.text_projection(txt_out.last_hidden_state)
        txt_cls = F.normalize(txt_tokens[:, 0, :], dim=-1)

        ref_out = trained_clip.vision_model(pixel_values=ref_tensor)
        ref_patches = trained_clip.visual_projection(ref_out.last_hidden_state)
        ref_cls = F.normalize(ref_patches[:, 0, :], dim=-1)

        composed_query = trained_combiner(ref_patches, txt_tokens, ref_cls, txt_cls).cpu().numpy().astype('float32')

        scores, indices = index.search(composed_query, k=int(top_k))

        retrieved_images = []
        for i, idx in enumerate(indices[0]):
            img_path = gallery_paths[int(idx)]
            actual_image = Image.open(img_path).convert('RGB')
            similarity = scores[0][i]
            retrieved_images.append((actual_image, f"Score: {similarity:.4f}"))

        return f"✅ Retrieved top {top_k} matches.", retrieved_images

    except Exception as e:
        return f"❌ ERROR: {e}", []

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("## 🧥 Multimodal Fashion Retrieval (Validation Only)")

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", label="Reference Base Garment")
            query_txt = gr.Textbox(placeholder="e.g., is sleeveless and has a darker floral print", label="Relative Text Modifier")
            k_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Number of Images to Show")
            submit_btn = gr.Button("Execute Search", variant="primary")
            status_msg = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_gallery = gr.Gallery(label="Retrieved Targets", columns=5, height="auto")

    submit_btn.click(
        fn=process_visual_query,
        inputs=[input_img, query_txt, k_slider],
        outputs=[status_msg, output_gallery]
    )

demo.launch(debug=True)

/tmp/ipykernel_2806/1424993050.py:38: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://474e23a00c1f115dff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
